Data Cleaning & Preprocessing

Importing Libraries

In [ ]:
import pandas as pd
import re
import emoji
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import opinion_lexicon
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
nltk.download('opinion_lexicon')


# Show all rows and columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Adjust width to avoid truncation
pd.set_option('display.max_colwidth', None)  # Show full column content

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!


Loading data into dataframe

In [62]:
df = pd.read_csv("../data/raw/Sentiment_Data.csv", encoding='latin1')
df.head()

,Tweet,Sentiment
0,"@_angelica_toy Happy Anniversary!!!....The Day the FreeDUMB Died (In the tune of Don McLean's ""American Pie"") #FreeDumbConvoy #Freedumbers #FluTruxKlan #convoywatch #convoy #FreedomConvoy https://t.co/ZT1cIPwmh9",Mild_Pos
1,"@McfarlaneGlenda Happy Anniversary!!!....The Day the FreeDUMB Died (In the tune of Don McLean's ""American Pie"") #FreeDumbConvoy #Freedumbers #FluTruxKlan #convoywatch #convoy #FreedomConvoy https://t.co/ZT1cIPwmh9",Mild_Pos
2,"@thevivafrei @JustinTrudeau Happy Anniversary!!!....The Day the FreeDUMB Died (In the tune of Don McLean's ""American Pie"") #FreeDumbConvoy #Freedumbers #FluTruxKlan #convoywatch #convoy #FreedomConvoy https://t.co/ZT1cIPwmh9",Mild_Pos
3,"@NChartierET Happy Anniversary!!!....The Day the FreeDUMB Died (In the tune of Don McLean's ""American Pie"") #FreeDumbConvoy #Freedumbers #FluTruxKlan #convoywatch #convoy #FreedomConvoy https://t.co/ZT1cIPwmh9",Mild_Pos
4,"@tabithapeters05 Happy Anniversary!!!....The Day the FreeDUMB Died (In the tune of Don McLean's ""American Pie"") #FreeDumbConvoy #Freedumbers #FluTruxKlan #convoywatch #convoy #FreedomConvoy https://t.co/ZT1cIPwmh9",Mild_Pos


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451332 entries, 0 to 451331
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Tweet      451331 non-null  object
 1   Sentiment  451332 non-null  object
dtypes: object(2)
memory usage: 6.9+ MB


Handling missing values

In [64]:
df.isnull().sum()

Tweet        1
Sentiment    0
dtype: int64

In [65]:
missing_rows = df[df.isnull().any(axis=1)]
print(missing_rows)

      Tweet Sentiment
75986   NaN   Neutral


In [66]:
df.dropna(inplace=True)

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 451331 entries, 0 to 451331
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Tweet      451331 non-null  object
 1   Sentiment  451331 non-null  object
dtypes: object(2)
memory usage: 10.3+ MB


In [68]:
df.isnull().sum()

Tweet        0
Sentiment    0
dtype: int64

Helper functions

In [69]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [70]:
df['clean_tweet'] = df['Tweet'].apply(clean_text)

In [71]:
def demojize_text(text):
    return emoji.demojize(text, delimiters=(" ", " "))

In [ ]:
df['clean_tweet'] = df['clean_tweet'].apply(demojize_text)

In [ ]:
slang_dict = {
    "idk": "i do not know",
    "brb": "be right back",
    "smh": "shaking my head",
    "tbh": "to be honest",
    "omg": "oh my god",
    "lol": "laughing out loud",
    "lmao": "laughing my ass off",
    "fyi": "for your information",
    "btw": "by the way",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "ikr": "i know right",
    "ttyl": "talk to you later",
    "bff": "best friends forever",
    "afk": "away from keyboard",
    "rofl": "rolling on the floor laughing",
    "np": "no problem",
    "ftw": "for the win",
    "jk": "just kidding",
    "ppl": "people",
    "ur": "your",
    "cya": "see you",
    "thx": "thanks",
    "gr8": "great",
    "pls": "please",
    "plz": "please"
}

def replace_slang(text, slang_dict= slang_dict):
    words = text.split()
    words = [slang_dict.get(w, w) for w in words]
    return " ".join(words)


In [ ]:
df['clean_tweet'] = df['clean_tweet'].apply(replace_slang)

In [ ]:
df['tokens'] = df['clean_tweet'].apply(word_tokenize)

In [ ]:

df['tokens'] = df['tokens'].apply(lambda x: [w for w in x if w not in stop_words])

In [ ]:
lemmatizer = WordNetLemmatizer()
df['tokens'] = df['tokens'].apply(lambda x: [lemmatizer.lemmatize(w) for w in x])

In [ ]:
df['clean_tweet'] = df['tokens'].apply((lambda x: ' '.join(x)))

Feature Engineering

In [ ]:
df['tweet_length'] = df['Tweet'].apply(len)

In [ ]:
df['num_hashtags'] = df['Tweet'].apply(lambda x: len(re.findall(r"#\w+", x)))

In [ ]:
pos_words = set(opinion_lexicon.positive())
neg_words = set(opinion_lexicon.negative())

In [ ]:
def lexicon_score(text, pos_words, neg_words):
    tokens = text.split()
    pos = sum(1 for w in tokens if w in pos_words)
    neg = sum(1 for w in tokens if w in neg_words)
    return pos - neg

df['lexicon_score'] = df['clean_tweet'].apply(lambda x: lexicon_score(x, pos_words, neg_words))


In [ ]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Sentiment'])

In [ ]:
label_encoder.classes_

array(['Mild_Neg', 'Mild_Pos', 'Neutral', 'Strong_Neg', 'Strong_Pos'],
      dtype=object)

In [ ]:
df.tail()

,Tweet,Sentiment,clean_tweet,tokens,tweet_length,num_hashtags,lexicon_score,label
451327,"Gaza; Peace n' Freedom - Viva Palestina convoy enters Algeria: \r\nThe Viva Palestina convoy, the massive relief c.. http://tinyurl.com/cajylt",Strong_Pos,gaza peace n freedom viva palestina convoy enters algeria viva palestina convoy massive relief c,"[gaza, peace, n, freedom, viva, palestina, convoy, enters, algeria, viva, palestina, convoy, massive, relief, c]",96,0,3,4
451328,"Face of Defense: Soldier Finds Freedom in U.S., Fights for Freedom in Iraq: CONVOY SUPPORT CEN... http://tinyurl.com/dyth62 RT @freerepublic",Strong_Pos,face defense soldier find freedom u fight freedom iraq convoy support cen rt,"[face, defense, soldier, find, freedom, u, fight, freedom, iraq, convoy, support, cen, rt]",76,0,3,4
451329,"Face of Defense: Soldier Finds Freedom in U.S., Fights for Freedom in Iraq: CONVOY SUPPORT CENTER SCANIA, Iraq, .. http://tinyurl.com/dyth62",Strong_Pos,face defense soldier find freedom u fight freedom iraq convoy support center scania iraq,"[face, defense, soldier, find, freedom, u, fight, freedom, iraq, convoy, support, center, scania, iraq]",88,0,3,4
451330,"Gaza; Peace n' Freedom - ""Israel stops aid convoy en route to Gaza from Hebron"": \r\nMa&quot;an News Agency report.. http://tinyurl.com/7mguhc",Strong_Pos,gaza peace n freedom israel stop aid convoy en route gaza hebron quot news agency report,"[gaza, peace, n, freedom, israel, stop, aid, convoy, en, route, gaza, hebron, quot, news, agency, report]",88,0,2,4
451331,@convoy 83 yes! get on freedom server!,Strong_Pos,yes get freedom server,"[yes, get, freedom, server]",22,0,1,4


In [ ]:
feature_cols = ['clean_tweet', 'tweet_length', 'num_hashtags', 'lexicon_score']
target_col = 'label'

X = df[feature_cols]
y = df[target_col]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

In [ ]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

y_train.to_frame().to_csv('../data/processed/y_train.csv', index=False)
y_val.to_frame().to_csv('../data/processed/y_val.csv', index=False)
y_test.to_frame().to_csv('../data/processed/y_test.csv', index=False)

In [ ]:
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)